# PUMA — Post-processing & Uncertainty Mapping for PEST++ IES

A complete walkthrough of the **puma** toolbox: discover a PEST++ IES run,
inspect it, and generate the full diagnostic + spatial figure suite — with
flexible control over iteration, layer, and observation grouping.

**Contents**
1. Setup & configuration
2. Load the run (`IesResults`) and inspect it
3. One-call full report
4. Convergence & objective function
5. Fit quality — grouped by observation type
6. Parameters & predictive uncertainty
7. Prior-data conflict & reliability
8. Time series (Marthe datetimes)
9. Spatial accuracy & uncertainty (Marthe / pymarthe)
10. Iteration & layer sweeps

> Every plotting function returns the path(s) it wrote and degrades gracefully
> if the data it needs is missing, so you can run any cell in isolation.

## 1. Setup & configuration

Install (once) and point at your run.

In [ ]:
# Install the toolbox (uncomment on first run)
# %pip install git+https://github.com/Ranveer82/Pestpp_Post.git
# Optional extras:
# %pip install "puma[spatial]"                                   # geopandas
# %pip install git+https://github.com/pymartheproject/pymarthe.git  # Marthe fields

In [ ]:
import os
from pathlib import Path
import matplotlib.pyplot as plt
from IPython.display import Image, display

import puma
from puma import IesResults, run_full_report, plots, spatial, timeseries
print("puma version:", puma.__version__)

In [ ]:
# ---- EDIT THESE -------------------------------------------------------------
PST_FILE = r"D:/m_329302/mrn_v11_v6.pst"   # your PEST++ IES control file
OUTPUT   = r"./PUMA_PLOTS_v6"                  # where figures are written
# Everything else (histo/pastp/mart/rma/config) is auto-detected from the
# .pst folder; set them explicitly below only to override.
# -----------------------------------------------------------------------------
OUTPUT = Path(OUTPUT); OUTPUT.mkdir(parents=True, exist_ok=True)

def show(paths, width=900):
    "Display one or many saved figures inline."
    if paths is None:
        print("(nothing produced)"); return
    if not isinstance(paths, (list, tuple)):
        paths = [paths]
    for p in paths:
        if p and Path(p).exists():
            display(Image(filename=str(p), width=width))

## 2. Load the run and inspect it

`IesResults` autonomously discovers the ensembles, phi files, iterations, and (for Marthe) the model files sitting next to the `.pst`.

In [ ]:
res = IesResults(PST_FILE)
res.posterior_iter = 3
res.prior_iter = 0
print(res.summary())

In [ ]:
# What Marthe model files did puma find next to the .pst?
found = res.discover_marthe_files()
for k, v in found.items():
    print(f"{k:8s}: {v}")

print("\nAvailable iterations :", res.available_iters)
print("Prior / posterior    :", res.prior_iter, "/", res.posterior_iter)
print("Observation groups   :", len(res.obs_groups()))
print("Parameter groups     :", res.par_groups())

## 3. One-call full report

The simplest way to get everything. Auto-detection handles the Marthe files, so this alone produces convergence, fit-by-type, parameter, uncertainty, time-series and spatial figures.

Use `obs_type_map` if your observation-group names don't match the `.histo` site names (classify by name prefix instead).

In [ ]:
written = run_full_report(
    PST_FILE,
    output_dir=OUTPUT,
    # iteration=6,               # which iteration is the "posterior" (default: last)
    # prior_iteration=0,         # baseline (default: 0)
    # field_layer=2,             # layer for the pymarthe field maps
    # field_max_reals=50,        # cap reconstructed realisations on big ensembles
    # obs_type_map={"Charge": ["c", "h", "w"], "Debit": ["d", "q"]},
    # make_timeseries=True,
)
print(f"\n{len(written)} figure(s) written to {OUTPUT}")

## 4. Convergence & objective function

Did the smoother converge, and did the whole ensemble of objective-function values collapse?

In [ ]:
show(plots.plot_phi_convergence(res, OUTPUT, log_scale=True))
show(plots.plot_phi_distribution(res, OUTPUT))
show(plots.plot_phi_by_group(res, OUTPUT))   # top groups by posterior phi

## 5. Fit quality — grouped by observation type

The central calibration-efficacy figure: measured vs simulated with the posterior credible interval, plus RMSE / NSE / R² / bias / PBIAS and CI coverage.

Pass the `.histo` (auto-found in `found['histo']`) to panel by observation type with correct units, or use `obs_type_map` for prefix-based grouping.

In [ ]:
import numpy as np
p = res.phi_actual()
real_mask = [v>0 for v in p.iloc[res.posterior_iter, 6:-1].values]
ac_idx = p.columns[6:-1][real_mask]
len(ac_idx)

In [ ]:
real_mask

In [ ]:
 len(p.iloc[res.posterior_iter, 6:-1].values)

In [ ]:
histo = found["histo"]                       # None if not a Marthe run
# Prefix-based grouping alternative (edit to your naming convention):
# type_map = {"Charge": ["c", "h", "w"], "Debit": ["d", "q"]}
type_map = None

show(plots.plot_one_to_one(res, OUTPUT, ci=(0.05, 0.95),
                           histo_file=histo, 
                           max_points=10000, # res.pst.nnz_obsall non zero obs from pst
                           obs_type_map=type_map))
show(plots.plot_residual_histograms(res, OUTPUT,
                                    histo_file=histo, obs_type_map=type_map))
show(plots.plot_residual_vs_simulated(res, OUTPUT,
                                      histo_file=histo, obs_type_map=type_map))

## 6. Parameters & predictive uncertainty

Prior vs posterior parameter distributions (log/native axis taken from the control file's `partrans`), how strongly the data informed each group, and predictive PDFs for any declared `++forecasts`.

In [ ]:
show(plots.plot_parameter_distributions(res, OUTPUT))
show(plots.plot_parameter_uncertainty_reduction(res, OUTPUT))
show(plots.plot_forecast_uncertainty(res, OUTPUT))   # skips if no forecasts

## 7. Prior-data conflict & reliability

Which observations the prior cannot reproduce (uses PEST++'s `*.pdc.csv` when present — fast, no ensemble load), and whether the posterior credible intervals are trustworthy (reliability diagram).

In [ ]:
show(plots.plot_prior_data_conflict(res, OUTPUT))


show(plots.plot_ensemble_coverage(res, 
                                  output_dir = OUTPUT,
                                  iteration = 3)) # by default res.posterior_iter

## 8. Time series (Marthe datetimes)

Per-site posterior ensemble band + measured points on a real calendar axis. For a Marthe model the datetimes are reconstructed from the `.pastp` / `.mart` — no date file needed.

In [ ]:
if found["pastp"]:
    obs_meta = puma.marthe.build_obs_meta(res, found["pastp"],
                                          mart_file=found["mart"])
    ts_paths = timeseries.plot_obs_timeseries(res, OUTPUT / "timeseries",
                                              obs_meta=obs_meta,
                                             max_sites=-1,  # to plot all ts 
                                             min_points = 2) # to plot all ts with at least 2 obs
    print(f"{len(ts_paths)} time-series figures")
    show(ts_paths[:3])          # preview the first few
else:
    print("No .pastp found — provide obs_meta_csv (obsnme,site,datetime) instead.")

## 9. Spatial accuracy & uncertainty (Marthe / pymarthe)

Where the calibration is good and where the posterior is still uncertain.
- **obs maps** need only the `.histo` (site x,y).
- **pilot-point maps** need a pilot-point file.
- **cell-by-cell field maps** (mixed pilot points + ZPC) reconstruct the posterior property field via the pymarthe `.config`.

In [ ]:
shape_files = [r"D:\Documents\kumar\BRGM\AP24ROU504_OPTIMISE - Dossier_projet - 06-Scientifique\MARTHE\Ranveer\Database\GIS\emprise_Marthe\Zone_modele_marthe_21-06-2023_repair.shp"]

In [ ]:
# (a) observation-location performance maps (accuracy / uncertainty / reliability)
if found["histo"]:
    show(spatial.plot_spatial_obs_performance(
        res, OUTPUT, histo_file=found["histo"], model_rma=found["rma"], shapefiles= shape_files))

In [ ]:
# (b) pilot-point posterior value / spread / uncertainty-reduction maps
PP_FILE = ["m_329302/pest/par/hk_pp_l01_z01.dat",
          "m_329302/pest/par/hk_pp_l02_z02.dat",
          "m_329302/pest/par/hk_pp_l03_z03.dat",
          "m_329302/pest/par/hk_pp_l03_z04.dat",
          "m_329302/pest/par/hk_pp_l03_z05.dat",
          "m_329302/pest/par/ss_pp_l01_z01.dat",
          "m_329302/pest/par/ss_pp_l02_z02.dat",
          "m_329302/pest/par/ss_pp_l03_z03.dat",
          "m_329302/pest/par/ss_pp_l03_z04.dat",
          "m_329302/pest/par/ss_pp_l03_z05.dat",
          "m_329302/pest/par/sy_pp_l01_z01.dat",
          "m_329302/pest/par/sy_pp_l02_z02.dat",
          "m_329302/pest/par/sy_pp_l03_z03.dat",
          "m_329302/pest/par/sy_pp_l03_z04.dat",
          "m_329302/pest/par/sy_pp_l03_z05.dat"]  # e.g. r"D:/m_325350/pest/par/hk_pp_l01_z12.dat"
if PP_FILE:
    for pp in PP_FILE:
        spatial.plot_pilotpoint_uncertainty(res, OUTPUT, pp_file=pp, shapefiles= shape_files)

In [ ]:
# (c) cell-by-cell posterior field statistics (needs pymarthe + the .config)
for l in [0,1,2]:
    for pr in ["permh", "emmca", "emmli"]:
        if found["config"]:
            show(spatial.plot_posterior_field_stats(
                res, OUTPUT, configfile=found["config"],
                prop=pr,          # property to reconstruct (None = first grid prop)
                layer=l,               # model layer to map
                max_reals=50))         # cap realisations for speed

In [ ]:
# (d) base property field on the grid with pilot-point uncertainty overlay
for l in [0,1,2]:
    for pr in ["permh", "emmca", "emmli"]:
        if found["rma"]:
            show(spatial.plot_marthe_property_field(
                res, OUTPUT, model_rma=found["rma"], prop=pr, layer=l,
                pp_file=PP_FILE))

## 10. Iteration & layer sweeps

Every plot accepts an `iteration` (and most a `prior_iteration`); the spatial field/obs functions also accept `layer`. Loop to compare how the fit and the field tighten across the smoother iterations or down the layers.

In [ ]:
# Compare the 1:1 fit at each available iteration
for it in res.available_iters:
    p = plots.plot_one_to_one(res, OUTPUT / f"iter_{it}",
                              iteration=it, histo_file=found["histo"])
    print(f"iteration {it}: {p}")

In [ ]:
# Posterior K-field std per layer (spatial uncertainty by depth)
for l in [0,1,2]:
    for pr in ["permh", "emmca", "emmli"]:
        
        if found["config"]:
            for lay in range(3):        # adjust to your number of layers
                spatial.plot_posterior_field_stats(
                    res, OUTPUT / f"layer_{lay}", configfile=found["config"],
                    prop=pr, layer=l, max_reals=200)

 length Mask = 17862, value = 17862
 length Mask = 12509, value = 12509
 length Mask = 275, value = 275
 length Mask = 4894, value = 4894
 length Mask = 7380, value = 7380
 length Mask = 17862, value = 17862
 length Mask = 12509, value = 12509
 length Mask = 275, value = 275
 length Mask = 4894, value = 4894
 length Mask = 7380, value = 7380
 length Mask = 17862, value = 17862
 length Mask = 12509, value = 12509
 length Mask = 275, value = 275
 length Mask = 4894, value = 4894
 length Mask = 7380, value = 7380
 length Mask = 17862, value = 17862
 length Mask = 12509, value = 12509
 length Mask = 275, value = 275
 length Mask = 4894, value = 4894
 length Mask = 7380, value = 7380
 length Mask = 17862, value = 17862
 length Mask = 12509, value = 12509
 length Mask = 275, value = 275
[puma] spatial: reconstructed permh for 123 realisation(s) over 349536 cells
[puma]   saved mrn_v11_v6_SPATIAL_fieldstat_permh_mean_L0.png
[puma]   saved mrn_v11_v6_SPATIAL_fieldstat_permh_std_L0.png
[puma]  

---
**Done.** All figures are under the `OUTPUT` folder. See `puma --help` for the equivalent command-line interface.